# v0.3.0 — Operational Governance e Monitoramento

**Projeto:** Renewable Energy MLOps — Wind Power Forecasting  
**Versão:** v0.3.0  
**Branch:** `ops/model-governance`  
**Status:** governança concluída; monitoramento operacional e Client/E2E validados; acompanhamento longitudinal iniciado  
**Champion anterior:** v10 — `d466ccb7fa294851a86a73b05d4e5735`  
**Challenger promovido:** v17 — `1d13a61244c54f06aa70f43a9993ea37`  
**OOT de decisão:** setembro/2026

Este notebook documenta a evolução do projeto de uma promoção baseada em métricas
históricas para um **Quality Gate com comparação Champion × Challenger sob o mesmo
conjunto Out-of-Time (same-OOT)**.

A v0.3 do projeto é composta por três frentes operacionais:

```text
v0.3
├── Operational Governance
├── Longitudinal Monitoring
└── Client / E2E Validation
```

Este notebook preserva a governança histórica e consolida o protocolo inicial de
monitoramento e a validação Client/E2E em 05/10/2026. O notebook principal permanece
como síntese curada da evolução do sistema. A v10 aparece somente na comparação
histórica de promoção; o monitoramento recente avalia exclusivamente a v17.

## 1. Objetivo

O objetivo desta etapa é tornar a promoção de modelos uma decisão operacional
**comparável, reproduzível e auditável**.

O problema identificado no lifecycle anterior era conceitual: o Challenger podia ser
avaliado no OOT atual enquanto o Champion era representado por uma métrica histórica
registrada em outra execução. Ainda que os dois números fossem válidos em seus
respectivos experimentos, eles não constituíam necessariamente uma comparação justa
para promoção automática.

A política da v0.3 passa a exigir:

- Champion e Challenger avaliados sobre o **mesmo OOT congelado**;
- mesma implementação de cálculo de MAE e nMAE;
- identidade explícita das versões e Runs;
- decisão automática associada a uma versão de política;
- metadados auditáveis persistidos no MLflow;
- promoção somente após o Quality Gate;
- hot reload do serviço somente depois da promoção.

## 2. Contexto experimental anterior

A v0.2.0 produziu duas linhas principais de evidência experimental:

1. **Simplificação do ensemble** — comparação controlada entre arquiteturas;
2. **Comparação de janelas de treinamento** — expanding, rolling 24 meses e rolling
   12 meses sob protocolo temporal comum.

No benchmark de janelas, a **expanding window** apresentou a melhor generalização OOT
entre os três tratamentos para setembro/2026. O modelo correspondente foi registrado
como **v17**.

Esse benchmark foi deliberadamente isolado do lifecycle operacional: nenhuma das três
janelas alterou automaticamente o alias `@champion`. Assim, a escolha experimental da
v17 como principal Challenger permaneceu separada da decisão de promoção.

## 3. Refatoração da governança

A principal mudança da v0.3 não foi adicionar uma nova arquitetura de Machine
Learning, mas corrigir **como modelos são comparados antes de uma promoção**.

### Antes

```text
Champion
└── métrica histórica da Run

Challenger
└── métrica da execução atual

        ↓

comparação potencialmente realizada
sob contextos OOT diferentes
```

### Depois

```text
Champion ─────┐
              ├── mesmo OOT congelado
Challenger ───┘
        ↓
shared evaluate_model_on_oot()
        ↓
automatic Quality Gate
        ↓
PROMOTE / REJECT
        ↓
auditable MLflow metadata
```

As melhorias implementadas nesta etapa incluem:

- extração de `evaluate_model_on_oot()` para um componente compartilhado de avaliação;
- alinhamento das features segundo o contrato esperado pelo modelo carregado;
- reutilização do mesmo contrato de avaliação pelo Monitoring;
- comparação same-OOT no Quality Gate;
- preservação de métricas históricas como evidência e baseline, sem utilizá-las como
  critério direto de promoção quando o OOT é diferente;
- `governance_policy_version = same_oot_v1`;
- registro do Champion anterior, Challenger, métricas same-OOT, deltas, OOT, decisão
  e motivo no MLflow;
- testes automatizados protegendo promoção, rejeição e regressões do comportamento
  same-OOT.

O escopo foi mantido deliberadamente simples. Não foi adicionada uma plataforma de
aprovação humana ou manual override, pois isso não resolve um problema concreto desta
fase do projeto.

## 4. Comparação definitiva — Champion v10 vs Challenger v17

A comparação operacional definitiva utiliza um único OOT congelado:

```text
OOT único
2026-09-01 03:00 → 2026-10-01 02:00
720 observações

   v10 (Champion) ─────┐
                       ├── evaluate_model_on_oot()
   v17 (Challenger) ───┘

Comparação controlada:
- mesmos dados
- mesmo target
- mesma capacidade
- mesma implementação de métricas
```

O intervalo temporal conceitual do mês é:

```text
[2026-09-01 03:00:00, 2026-10-01 03:00:00)
```

Como os dados são horários, a última observação presente é
`2026-10-01 02:00:00`.

### 4.1. Configuração reproduzível

A célula abaixo fixa explicitamente as duas versões e o snapshot OOT. Ela não utiliza
o alias `@champion` para carregar os modelos avaliados, evitando que uma mudança futura
do alias altere retrospectivamente o experimento.
> Seção histórica: esta comparação já foi realizada e não é requisito para executar
> o monitoramento atual. Para inspecionar outubro, use a seção 11 independente.


In [1]:
import os

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.tracking import MlflowClient

from energy_mlops.config import settings
from energy_mlops.data.feature_utils import select_model_features
from energy_mlops.models.evaluation import evaluate_model_on_oot

MODEL_NAME = "ensemble_lgb_xgb_rf_bahia"

CHAMPION_VERSION = "10"
CHAMPION_RUN_ID = "d466ccb7fa294851a86a73b05d4e5735"

CHALLENGER_VERSION = "17"
CHALLENGER_RUN_ID = "1d13a61244c54f06aa70f43a9993ea37"

OOT_DATASET = "oot_test_wind_energy_2026_09.parquet"
OOT_PATH = (
    f"s3://{settings.RUSTFS_BUCKET}/gold/"
    f"{OOT_DATASET}"
)

os.environ["AWS_ACCESS_KEY_ID"] = settings.RUSTFS_ROOT_USER
os.environ["AWS_SECRET_ACCESS_KEY"] = settings.RUSTFS_ROOT_PASSWORD
os.environ["MLFLOW_S3_ENDPOINT_URL"] = settings.RUSTFS_ENDPOINT
os.environ["AWS_ENDPOINT_URL"] = settings.RUSTFS_ENDPOINT

mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)

client = MlflowClient(
    tracking_uri=settings.MLFLOW_TRACKING_URI
)

/home/wanderson/.cache/pypoetry/virtualenvs/energy-mlops-G3NdcX_6-py3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 4.2. Identidade dos modelos e validação do OOT

As versões são verificadas contra os `run_id` congelados. O OOT também é validado em
número de linhas e limites temporais antes da avaliação.

In [2]:
champion_info = client.get_model_version(
    MODEL_NAME,
    CHAMPION_VERSION,
)

challenger_info = client.get_model_version(
    MODEL_NAME,
    CHALLENGER_VERSION,
)

assert champion_info.run_id == CHAMPION_RUN_ID
assert challenger_info.run_id == CHALLENGER_RUN_ID

df_oot = pd.read_parquet(
    OOT_PATH,
    storage_options=settings.storage_options,
)

dates = pd.to_datetime(
    df_oot["date"],
    errors="raise",
)

assert len(df_oot) == 720
assert dates.min() == pd.Timestamp("2026-09-01 03:00:00")
assert dates.max() == pd.Timestamp("2026-10-01 02:00:00")

X_oot = select_model_features(df_oot)

print("OOT:", OOT_DATASET)
print("rows:", len(df_oot))
print("start:", dates.min())
print("end:", dates.max())
print("features:", X_oot.columns.tolist())

OOT: oot_test_wind_energy_2026_09.parquet
rows: 720
start: 2026-09-01 03:00:00
end: 2026-10-01 02:00:00
features: ['temperature_2m', 'wind_speed_100m', 'wind_direction_100m', 'wind_temp_ratio', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'wind_speed_roll_mean_3h']


### 4.3. Avaliação same-OOT

Os dois modelos são carregados por **versão explícita** e avaliados pela mesma função.

In [3]:
champion_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}/{CHAMPION_VERSION}"
)

challenger_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}/{CHALLENGER_VERSION}"
)

champion_eval = evaluate_model_on_oot(
    champion_model,
    X_oot,
    df_oot,
)

challenger_eval = evaluate_model_on_oot(
    challenger_model,
    X_oot,
    df_oot,
)

mae_delta_mw = (
    challenger_eval.mae_mw
    - champion_eval.mae_mw
)

nmae_delta_pp = (
    challenger_eval.nmae_pct
    - champion_eval.nmae_pct
)

mae_improvement_pct = (
    (
        champion_eval.mae_mw
        - challenger_eval.mae_mw
    )
    / champion_eval.mae_mw
    * 100
)

### 4.4. Tabela comparativa

A saída abaixo é gerada diretamente das avaliações same-OOT e pode ser reproduzida
sempre que os artifacts originais permanecerem disponíveis.

In [4]:
comparison = pd.DataFrame(
    {
        "Model": [
            "v10 — Champion anterior",
            "v17 — Challenger",
        ],
        "MAE (MW)": [
            champion_eval.mae_mw,
            challenger_eval.mae_mw,
        ],
        "nMAE (%)": [
            champion_eval.nmae_pct,
            challenger_eval.nmae_pct,
        ],
    }
)

comparison["Δ MAE vs v10 (MW)"] = (
    comparison["MAE (MW)"]
    - champion_eval.mae_mw
)

comparison["Δ nMAE vs v10 (p.p.)"] = (
    comparison["nMAE (%)"]
    - champion_eval.nmae_pct
)

display(
    comparison.style.format(
        {
            "MAE (MW)": "{:.3f}",
            "nMAE (%)": "{:.4f}",
            "Δ MAE vs v10 (MW)": "{:+.3f}",
            "Δ nMAE vs v10 (p.p.)": "{:+.4f}",
        }
    )
)

print(
    "MAE relative improvement:",
    f"{mae_improvement_pct:.4f}%"
)

,Model,MAE (MW),nMAE (%),Δ MAE vs v10 (MW),Δ nMAE vs v10 (p.p.)
0,v10 — Champion anterior,803.372,6.8162,+0.000,+0.0000
1,v17 — Challenger,787.184,6.6788,-16.189,-0.1374


MAE relative improvement: 2.0151%


### 4.5. Resultado observado

A execução real do protocolo produziu:

| Métrica | v10 — Champion anterior | v17 — Challenger | Δ v17 − v10 |
|---|---:|---:|---:|
| MAE (MW) | 803.3723 | **787.1836** | **−16.1887 MW** |
| nMAE (%) | 6.8162 | **6.6788** | **−0.1374 p.p.** |

A redução relativa do MAE foi de aproximadamente **2.0151%**.

A métrica histórica da v10 registrada em sua Run era `829.5865 MW`. Quando a v10 foi
reavaliada no mesmo OOT da v17, seu MAE foi `803.3723 MW`. Essa diferença demonstra
por que métricas históricas de contextos distintos não devem, por si só, decidir uma
promoção automática.

A v17 reproduziu `787.1836 MW` e `6.6788%`, consistentes com o benchmark original da
expanding window, fornecendo uma verificação adicional de reprodutibilidade.

**Conclusão:** sob o OOT congelado de setembro/2026 e a política `same_oot_v1`, a v17
satisfaz o critério primário `MAE_IMPROVED_SAME_OOT`.

Este resultado não implica superioridade universal da v17; a evidência é específica
ao protocolo e ao OOT avaliados.

## 5. Promoção operacional auditável

Depois da comparação read-only, a v17 foi submetida ao lifecycle operacional real.

O Quality Gate reavaliou Champion e Challenger no mesmo OOT e registrou:

```text
quality_gate_status = PROMOTED_CHAMPION
governance_final_decision = PROMOTE
governance_decision_reason = MAE_IMPROVED_SAME_OOT
governance_policy_version = same_oot_v1
```

Metadados principais persistidos na Run da v17:

| Campo | Valor |
|---|---|
| Champion anterior | v10 |
| Challenger | v17 |
| Champion MAE same-OOT | 803.372317 MW |
| Challenger MAE same-OOT | 787.183586 MW |
| Δ MAE | −16.188731 MW |
| Champion nMAE same-OOT | 6.816154% |
| Challenger nMAE same-OOT | 6.678802% |
| Δ nMAE | −0.137352 p.p. |
| OOT | `oot_test_wind_energy_2026_09.parquet` |
| OOT rows | 720 |
| Política | `same_oot_v1` |

A mudança operacional seguiu o fluxo:

```text
v10 @champion
      ↓
same-OOT Quality Gate
      ↓
v17 MAE_IMPROVED_SAME_OOT
      ↓
auditable governance metadata
      ↓
MLflow alias @champion → v17
      ↓
hot reload
      ↓
API serving v17
```

O hot reload respondeu com sucesso e a API passou a reportar a v17 como modelo ativo.

### 5.1. Inspeção read-only do estado atual

Esta célula **não promove modelos**. Ela apenas consulta o Registry e os metadados
persistidos para auditoria.

In [5]:
champion_now = client.get_model_version_by_alias(
    MODEL_NAME,
    "champion",
)

promoted_run = client.get_run(
    CHALLENGER_RUN_ID
)

governance_keys = [
    "quality_gate_status",
    "governance_final_decision",
    "governance_decision_reason",
    "governance_policy_version",
    "governance_previous_champion_version",
    "governance_challenger_version",
    "governance_champion_mae_mw_same_oot",
    "governance_challenger_mae_mw_same_oot",
    "governance_mae_delta_mw_same_oot",
    "governance_champion_nmae_pct_same_oot",
    "governance_challenger_nmae_pct_same_oot",
    "governance_nmae_delta_pp_same_oot",
    "governance_oot_dataset",
    "governance_oot_rows",
    "governance_oot_start_utc",
    "governance_oot_end_utc",
]

governance_state = {
    key: promoted_run.data.tags.get(key)
    for key in governance_keys
}

print(
    "Current @champion:",
    f"v{champion_now.version}",
    champion_now.run_id,
)

pd.Series(
    governance_state,
    name="value",
).to_frame()

Current @champion: v17 1d13a61244c54f06aa70f43a9993ea37


,value
quality_gate_status,PROMOTED_CHAMPION
governance_final_decision,PROMOTE
governance_decision_reason,MAE_IMPROVED_SAME_OOT
governance_policy_version,same_oot_v1
governance_previous_champion_version,10
governance_challenger_version,17
governance_champion_mae_mw_same_oot,803.3723172253395
governance_challenger_mae_mw_same_oot,787.1835861175788
governance_mae_delta_mw_same_oot,-16.18873110776076
governance_champion_nmae_pct_same_oot,6.816153646397424


### 5.2. Evidência do serving

Após o hot reload, `/model-info` retornou:

```json
{
  "model_name": "ensemble_lgb_xgb_rf_bahia",
  "alias": "champion",
  "version": 17,
  "run_id": "1d13a61244c54f06aa70f43a9993ea37",
  "metrics": {
    "oot_mae_mw": 787.1835861175788,
    "oot_nmae_pct": 6.678801541769502,
    "oot_mae_fc_pct": 6.678801541769504,
    "oot_r2_score": 0.8613955426061799
  }
}
```

Isso fecha a cadeia entre decisão de governança, Model Registry e estado servido pela
API.

## 6. Conclusões da etapa de Operational Governance

A principal conclusão desta etapa não é apenas que a v17 apresentou menor MAE.

A contribuição operacional da v0.3 foi estabelecer um **protocolo de promoção**:

1. escolher um Challenger com base em evidência experimental;
2. congelar e identificar o OOT de decisão;
3. reavaliar Champion e Challenger nas mesmas condições;
4. aplicar uma política explícita de Quality Gate;
5. persistir a decisão e sua proveniência;
6. alterar o alias somente depois da aprovação;
7. atualizar o serving por hot reload;
8. verificar que a API está servindo a versão promovida.

No OOT de setembro/2026, a v17 reduziu o MAE em `16.1887 MW` e o nMAE em
`0.1374 p.p.` em relação à v10. A promoção ocorreu pelo critério
`MAE_IMPROVED_SAME_OOT`.

A v17 passa, portanto, a ser o Champion operacional do projeto para a próxima etapa da
v0.3.

## 7. Transição da governança para a operação

A promoção v17 encerrou a decisão same-OOT. A etapa seguinte foi validar o serving,
a ingestão parcial e o monitoramento com dados realmente disponíveis em outubro,
sem depender da v10 nem pressupor um novo mês completo.

As seções seguintes documentam o protocolo, a primeira janela real e as fronteiras
Client/E2E exercitadas. Longitudinalidade significa acompanhar novas janelas ao longo
do tempo; reexecutar a mesma janela para validar gráficos não constitui essa evidência.


## 8. Protocolo inicial de monitoramento — v0.3

**Data de consolidação:** 05/10/2026. O identificador editorial deste protocolo é
`monitoring_observational_v1`; ele ainda não é uma tag persistida automaticamente
pelo código. A política de promoção permanece `same_oot_v1`.

O monitoramento operacional começa pela disponibilidade real dos dados, e não por
uma estrutura mensal presumida. Setembro é a referência OOT de seleção/promoção;
outubro é a primeira janela recente observada nesta validação. A v10 permanece
somente como evidência histórica de governança, sem participar do monitoramento atual.

| Fronteira | Regra |
|---|---|
| Datas | Dias locais encerrados; persistência/identidade temporal em UTC e apresentação em UTC−03 |
| Dados | Snapshot não vazio, ordenado, timestamps únicos na grade horária e features finitas |
| Features | Preservar engenharia do Gold; contexto das duas horas anteriores para rolling 3h |
| Ground truth | Não imputar geração; zero observado é válido, ausência não é zero |
| Horas avaliáveis | Geração finita e não negativa; capacidade finita e positiva |
| Modelo | Resolver alias uma vez e avaliar por versão/run explícitos |
| Data drift | Nove features e fração sinalizada; calendário interpretado separadamente |
| Performance | Apenas horas avaliáveis; cobertura e janela sempre registradas |
| Alerta inicial | Share de drift ≥50%; aumento de nMAE ≥2 p.p. na janela agregada |
| Diagnóstico diário | Descritivo; não aplicar automaticamente o limiar agregado aos dias |
| Treinamento | Padrão observacional; CT exige pedido explícito, mês completo, truth integral e algum sinal de drift |
| Promoção | Decisão independente, subordinada ao Quality Gate same-OOT |
| Evidência | HTML, JSON, CSV horário e diário com identidade da execução |

O nMAE segue o contrato compartilhado:

$$nMAE = 100\,\frac{MAE_{MW}}{\max(capacidade_{MW})}$$

A capacidade máxima é calculada entre as horas válidas da janela avaliada. No
snapshot de outubro, ela foi 11.786,3 MW. A mesma fórmula é aplicada dentro de cada
dia para os diagnósticos diários. `100 − nMAE` não representa taxa de acerto.

Os limiares são regras operacionais iniciais, não limites calibrados a partir de
uma série longitudinal extensa. Alertas em janelas parciais motivam investigação;
não justificam, sozinhos, retreinamento ou afirmações de degradação persistente.

### 8.1. Natureza da avaliação

A comparação atual é **estimativa do modelo com meteorologia observada/reanálise ×
geração observada ONS**. O modelo não foi ajustado às 72 horas usadas nesta avaliação,
mas o teste tampouco é um replay das previsões day-ahead emitidas anteriormente.

Para avaliar day-ahead, será necessário guardar timestamp de emissão, horizonte,
payload meteorológico de previsão, versão/run do modelo e resultado efetivamente
emitido, pareando-o posteriormente com a geração observada. Essa frente permanece
pendente; não é substituída pela disponibilidade de gráficos retrospectivos.


## 9. Primeira janela real — 1 a 4 de outubro de 2026

A ingestão parcial terminou em `Completed` e salvou:

` s3://energy-lake/monitoring/current_2026-10-01_2026-10-04.parquet `

O snapshot mensal Gold não foi alterado. A geração de features utilizou as duas
horas imediatamente anteriores do snapshot canônico para contextualizar a rolling.

| Janela | Limites locais (UTC−03) | Limites UTC | Horas |
|---|---|---|---:|
| Referência OOT | 01/09 00:00 → 30/09 23:00 | 01/09 03:00 → 01/10 02:00 | 720 |
| Meteorologia current | 01/10 00:00 → 04/10 23:00 | 01/10 03:00 → 05/10 02:00 | 96 |
| Ground truth válido | 01/10 00:00 → 03/10 23:00 | 01/10 03:00 → 04/10 02:00 | 72 |

Todos os limites acima são de 2026 e representam observações horárias inclusivas.
O snapshot apresentou zero duplicatas e zero horas ausentes no intervalo observado.
Em 04/10, havia meteorologia para 24 horas, mas nenhuma geração observada no snapshot
extraído. Isso caracteriza atraso/ausência de truth nessa extração, não geração zero.

### 9.1. Identidade e artefatos

| Campo | Valor |
|---|---|
| Modelo | `ensemble_lgb_xgb_rf_bahia`, v17 |
| Run | `1d13a61244c54f06aa70f43a9993ea37` |
| TRAIN end registrado | `2026-09-01 02:00:00` UTC |
| Primeira execução | `20261005T130632Z-cf35b098` |
| Execução com gráficos horários | `20261005T151634Z-06d0efb9` |
| Avaliação da execução com gráficos | 05/10/2026 12:16:37, aproximadamente, UTC−03 |
| Ground truth | `PARTIAL_GROUND_TRUTH`, 72/96 horas (75%) |
| Janela | `PARTIAL_WINDOW` |
| Treinamento | `training_requested=false`, `training_eligible=false`, `NOT_TRIGGERED` |

A segunda execução avaliou o mesmo snapshot/modelo e reproduziu o mesmo nMAE; ela
valida a persistência dos CSVs e a integração com o dashboard. **Não é uma segunda
janela temporal independente**, nem evidência adicional de persistência do drift.

Prefixo histórico da execução com gráficos:

`energy-lake/monitoring/drift_report/20261005T151634Z-06d0efb9`

Arquivos: `.html`, `.json`, `.hourly.csv`, `.daily.csv`. O caminho
`monitoring/drift_report.html` permanece como cópia de compatibilidade; o dashboard
atual usa o JSON histórico mais recente e os artefatos pareados daquela execução.

### 9.2. Resultados agregados e interpretação de drift

| Indicador | Resultado |
|---|---:|
| Baseline OOT de setembro | 6,6788% |
| nMAE current, 72 horas válidas | 7,6642% |
| Aumento | +0,9854 p.p. |
| Limiar de performance | +2,0 p.p. |
| Performance drift segundo a regra | Não detectado |
| Data drift | 7/9 features, 77,78%; detectado |

A comparação é entre um mês de referência e uma janela curta. As horas possuem
dependência temporal; os p-valores do relatório não são tratados como prova de
causalidade, de sete evidências independentes ou de degradação persistente.

| Features | Resultado | Leitura |
|---|---|---|
| `hour_sin`, `hour_cos` | Sem drift | Janelas com dias completos preservam distribuição horária |
| `month_sin`, `month_cos` | Drift | Passagem esperada de setembro para outubro |
| `temperature_2m`, `wind_speed_100m`, `wind_direction_100m` | Drift | Mudanças meteorológicas na janela observada |
| `wind_temp_ratio`, `wind_speed_roll_mean_3h` | Drift | Derivações correlacionadas às variáveis meteorológicas |

O share global é preservado como diagnóstico do contrato completo. Não se muda o
limiar nem se retira uma feature do modelo para silenciar esta execução.

### 9.3. Diagnóstico diário

| Dia | Horas truth | MAE (MW) | nMAE (%) | Viés médio (MW) |
|---|---:|---:|---:|---:|
| 01/10 | 24 | 1.042,5587 | 8,8455 | −1.042,5587 |
| 02/10 | 24 | 602,1446 | 5,1089 | −341,6745 |
| 03/10 | 24 | 1.065,2861 | 9,0383 | +456,1950 |
| 04/10 | 0 | Indisponível | Indisponível | Indisponível |

Erro assinado = estimativa − observado. Em 01/10, as 24 horas apresentaram
subestimação; em 02/10 predominou subestimação; em 03/10 houve alternância de sinais.
O viés pode esconder compensações e não substitui o MAE.

Os dias 1 e 3 excedem numericamente +2 p.p. contra a baseline mensal, mas esse limiar
não foi definido para decisão diária. Em setembro, a mediana diária foi 6,0392%, o
P90 foi 9,4284% e o máximo 11,7517%. Seis dos 30 dias tiveram nMAE ≥8,8455%; cinco
 tiveram nMAE ≥9,0383%. Portanto, esses erros possuem precedentes na referência,
sem que isso demonstre ausência de problema futuro.

### 9.4. Mudança de regime meteorológico

No current completo (96 horas), a temperatura média passou de 21,0717 para 22,6141 °C;
a velocidade média armazenada passou de 25,0514 para 23,5374; a direção média circular
passou de 104,4558° para 95,9176°. A concentração circular aumentou de 0,9562 para
0,9858. A direção é circular e não deve ser resumida por média linear entre 0° e 360°.

Não se atribui unidade física à velocidade nessa comparação sem confirmar o contrato
do extrator. A reconciliação dos rótulos/unidades entre Archive e forecast deve ser
verificada antes de qualquer conversão. Os números foram comparados na escala
armazenada, sem converter inputs do modelo.

Os dias 1 e 2 tiveram velocidades médias próximas (21,1825 e 20,8854), mas erros
bem diferentes. A redução do vento, isoladamente, não explica o padrão observado.


## 10. Validação Client/E2E e produto de monitoramento

### 10.1. Software lifecycle e serving

A imagem foi construída/publicada via GitHub Actions e atualizada manualmente no KinD
por digest publicado no GHCR:

`sha256:036d3cc5955b9bd931c4fdba0636b52840f766bf94e9b140576727fe07da2cb1`

Revisão OCI: `267ea7ac0467df097574656144e0eaa2f1a21b44`.

O rollout terminou com sucesso e `make validate` confirmou workloads Ready e endpoints
locais esperados. `/model-info` confirmou v17 e o mesmo run da governança.
O erro de inferência por checkpoint de capacidade ausente foi corrigido incluindo
`data/reference/bahia_wind_capacity_checkpoints.csv` na imagem e exigindo arquivo não
vazio no build. O CSV já era versionado; `.dockerignore` recebeu a exceção específica.

O smoke HTTP em `/predict/batch`, com timestamp `2026-10-04T12:00:00Z`, retornou
`predicted_fc=0.17757323076511358` e `predicted_mw=2092.931369766858`.
O botão de previsão do Streamlit também foi exercitado com sucesso. Esses resultados
validam contrato e execução, não qualidade futura das previsões day-ahead.

A nova funcionalidade de monitoring foi executada pelo ambiente Poetry local,
consumindo MLflow/RustFS no KinD; o dashboard também roda localmente. Não se afirma
que as alterações posteriores dos gráficos já estejam publicadas nessa imagem.

### 10.2. Dashboard e pacote de evidência

A aba apresenta janelas/fuso, versão/run, cobertura de truth, nMAE, delta e estado de
retreinamento. Plotly exibe a comparação horária, barras de erro e filtro por dia.
A geração observada fica com lacuna em 04/10; não é imputada. O resumo diário mostra
MAE, nMAE e viés. O relatório Evidently permanece expansível.

O botão Atualizar monitoramento limpa o cache. O download ZIP contém:

- `relatorio.html`: relatório interativo de data drift;
- `resumo.json`: identidade, cobertura, métricas e caminhos;
- `comparacao_horaria.csv`: estimativa, observado, capacidade, flags e erros;
- `resumo_diario.csv`: métricas e cobertura por dia.

O dashboard lê artefatos, sem carregar o modelo ou recalcular inferência. O batch
preserva a tarefa de performance existente e realiza uma segunda carga/inferência
da mesma versão para persistir a série completa, inclusive horas sem truth. O nMAE
derivado do CSV é comparado ao valor registrado antes de publicar o resumo.

### 10.3. Verificações realizadas

- Suíte via Tox/Python 3.14: **154 testes aprovados**, 2 warnings internos Evidently/NumPy.
- Ruff: um import fora de ordem foi corrigido com `--fix`; o conteúdo da correção
  não modifica lógica. A documentação não inventa uma saída posterior de Ruff.
- O usuário executou `git diff --check` sem erro apresentado.
- Ingestão e monitoramento reais concluídos; visualização e download exercitados.
- ZIP baixado conferido: 96 linhas, 72 truth, identidade única de execução/run,
  lacunas preservadas e nMAE recalculado de 7,66423540874785%.

O total de testes não é percentual de coverage. A validação cobre os cenários
exercitados, não todos os cenários de falha ou de operação futura.


## 11. Inspeção read-only da evidência exportada

As células seguintes são independentes das células históricas v10/v17 acima.
Elas leem o ZIP exportado da execução validada e não acionam ingestão, treinamento,
promoção ou reload. Ajuste `EVIDENCE_ZIP` para o arquivo local. O ZIP é evidência
externa, não dataset versionado no Git. As novas células não foram executadas contra
o cluster pelo editor; resultados congelados acima vieram dos artefatos conferidos.


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd

EVIDENCE_ZIP = Path("../validation_20261005/monitoramento_20261005T151634Z-06d0efb9.zip")
with zipfile.ZipFile(EVIDENCE_ZIP) as evidence:
    monitoring_summary = json.loads(evidence.read("resumo.json"))
    monitoring_hourly = pd.read_csv(evidence.open("comparacao_horaria.csv"))
    monitoring_daily = pd.read_csv(evidence.open("resumo_diario.csv"))

assert monitoring_summary["execution_id"] == "20261005T151634Z-06d0efb9"
assert monitoring_summary["model"]["version"] == "17"
assert monitoring_summary["model"]["run_id"] == "1d13a61244c54f06aa70f43a9993ea37"
assert monitoring_hourly["execution_id"].eq(monitoring_summary["execution_id"]).all()
assert monitoring_hourly["model_run_id"].eq(monitoring_summary["model"]["run_id"]).all()
assert len(monitoring_hourly) == 96
assert monitoring_hourly["truth_valid"].sum() == 72
valid_hourly = monitoring_hourly.loc[monitoring_hourly["truth_valid"]]
replayed_nmae = 100 * valid_hourly["absolute_error_mw"].mean() / valid_hourly["capacidade_mw"].max()
assert np.isclose(replayed_nmae, monitoring_summary["performance"]["current_nmae_pct"])
assert monitoring_hourly.loc[~monitoring_hourly["truth_valid"], ["observed_mw", "error_mw"]].isna().all().all()
print("nMAE reproduzido:", replayed_nmae)
display(monitoring_daily)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

local_time = pd.to_datetime(monitoring_hourly["date"], utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)
figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                      subplot_titles=["Estimativa com meteorologia observada × geração observada", "Erro: estimativa − observado"])
for column, label, color in [("predicted_mw", "Estimativa", "#38BDF8"),
                             ("observed_mw", "Observado ONS", "#10B981")]:
    figure.add_trace(go.Scatter(x=local_time, y=monitoring_hourly[column], name=label,
                               connectgaps=False, line=dict(color=color)), row=1, col=1)
figure.add_trace(go.Bar(x=local_time, y=monitoring_hourly["error_mw"], name="Erro",
                       marker_color=np.where(monitoring_hourly["error_mw"] >= 0, "#F59E0B", "#A78BFA")), row=2, col=1)
figure.add_hline(y=0, row=2, col=1)
figure.update_yaxes(title_text="Potência (MW)", row=1, col=1)
figure.update_yaxes(title_text="Erro (MW)", row=2, col=1)
figure.update_xaxes(title_text="Horário local (UTC−03)", row=2, col=1)
figure.update_layout(template="plotly_dark", height=650, hovermode="x unified")
figure.show()


## 12. Conclusão e pendências da v0.3

| Frente | Estado em 05/10/2026 |
|---|---|
| Operational Governance | Concluída: promoção same-OOT e serving v17 verificados |
| Monitoramento operacional | Validado em janela real, com truth parcial, artefatos e dashboard |
| Longitudinal monitoring | Iniciado; persistência temporal ainda não estabelecida |
| Client/E2E | Smoke API, botão Streamlit e consumo/download de monitoring validados |
| Avaliação day-ahead observada | Pendente de captura das previsões emitidas e pareamento posterior |
| Cloud/IaC v1.0 | Etapa futura |

Prioridades: incorporar truth atrasado gerando nova avaliação; acompanhar novas
janelas identificadas sem contar reexecuções como amostras independentes; verificar
unidades Archive/forecast; preservar snapshots por execução ou adicionar versão/hash
para replay rigoroso; revisar limiares quando houver evidência longitudinal suficiente.

Os snapshots current ainda usam caminhos derivados das datas e podem ser substituídos
por uma nova ingestão. Os CSVs históricos fixam a comparação exibida, mas os caminhos
de datasets no JSON não garantem imutabilidade do dado-fonte. Essa limitação é explícita.

O dashboard atualmente seleciona a execução JSON mais recente; navegação entre
execuções e alertas de staleness são melhorias possíveis, não requisitos para validar
a primeira janela. A otimização da imagem Docker foi adiada para depois desta etapa,
antecedendo Cloud/IaC/deployment reproduzível da v1.0.

A conclusão válida é que a cadeia operacional funciona e expõe evidência auditável
para as condições exercitadas. Não se conclui estabilidade futura nem se encerra a
frente longitudinal com base em três dias de truth.
